# Phase 03 - Classifier core

Datasets and transforms, model builders (ResNet50, ConvNeXt-Tiny, Swin-Tiny), the
train/evaluate loop and run folders. See `docs/instructions/03-classifier-core.md`.

Cells whose first line is `#| export` are copied, in order, into the
generated `group3-starvenus.py` by `tools/export_py.py`. Keep outputs cleared before committing.

The experiment registry (`EXPERIMENTS`) and the choices file live in `90_cli.ipynb`; this notebook is
the engine that the registry drives.

## Experiment configuration (T3.1)

One `ExperimentConfig` describes one row of the experiment matrix (README section 5). The engine below
is generic: it never looks at `exp_id`, only at these fields.

Three fields can be `None` in a registry entry and are then filled in **at run time** from
`_work/results/choices.json` (written by the gates of phase 04): `input_mode` ("best of E0"),
`backbone` ("best of E2") and `lr` (per backbone, from the split-0 sweep). A resolved config has none
of them left empty.

`select_domains` always equals `train_domains`: the checkpoint is chosen on the validation images of
the domains the model was trained on. A white-only model (W->F rows) is therefore chosen on **white**
validation images; choosing on field would leak field information into a "white-only" model.

In [ ]:
#| export
import gc
import math
import platform
import random
import re
import time
from urllib.parse import urlparse

import torch
import torch.nn.functional as F
import torchvision
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

NUM_CLASSES = len(CLASS_NAMES)
BACKBONES = ("resnet50", "convnext_tiny", "swin_t")
INPUT_MODES = ("squash", "pad", "rect")
# (height, width) of the network input at an equal pixel budget (~100k px): E0 compares geometry, not resolution.
INPUT_SIZES = {"squash": (320, 320), "pad": (320, 320), "rect": (448, 224)}
SUBSETS = ("white", "field", "mixed")
KINDS = ("classifier", "segmenter", "analysis")


class ExperimentError(Exception):
    """Base class of errors that the CLI reports as one clean line (no traceback)."""


class UnknownExperimentError(ExperimentError):
    pass


class ChoicesError(ExperimentError):
    """A value that a gate should have written to choices.json is missing."""


class NotReadyError(ExperimentError):
    """The experiment needs a later phase that is not implemented yet."""


@dataclass(frozen=True)
class ExperimentConfig:
    exp_id: str
    backbone: str | None  # None: the winning backbone of gate G1 (choices.json "backbone")
    train_domains: tuple[str, ...]
    select_domains: tuple[str, ...]  # must equal train_domains
    test_subsets: tuple[str, ...]  # which test subsets the report uses (all three are always evaluated)
    input_mode: str | None  # None: the winning input mode of gate G0 (choices.json "input_mode")
    use_composites: bool = False  # Method A: extra training samples from CompositeSource (phase 05)
    seg_crop: bool = False  # Method B: input_hook crops to the segmenter box (phase 06)
    kind: str = "classifier"  # "segmenter" / "analysis" rows exist in the matrix but are not run here
    alias_of_winner: str = ""  # "E0": this row is the winning E0 row (same run folder, no retraining)
    # Recipe (section 6.1 of the report). Defaults are the T3.4 table.
    stage1_epochs: int = 2  # linear probe: backbone frozen, head only
    stage1_lr: float = 1e-3
    epochs: int = 30  # stage 2: fine-tune everything
    lr: float | None = None  # None: choices.json lr[backbone] (split-0 sweep over SWEEP_LRS)
    weight_decay: float = 0.05
    warmup_epochs: int = 1
    label_smoothing: float = 0.1
    batch_size: int = 32
    grad_accum: int = 1
    clip_norm: float = 1.0
    patience: int = 10  # stage-2 epochs without improvement of the selection metric
    dropout: float = 0.2
    pretrained: bool = True
    size_div: int = 1  # dev/smoke only: divide the input size (448x224 -> 112x56 with 4)
    note: str = ""

    def __post_init__(self):
        if self.kind not in KINDS:
            raise ValueError(f"{self.exp_id}: kind must be one of {KINDS}, got {self.kind!r}")
        if self.backbone is not None and self.backbone not in BACKBONES and self.kind == "classifier":
            raise ValueError(f"{self.exp_id}: unknown backbone {self.backbone!r}")
        if self.input_mode is not None and self.input_mode not in INPUT_MODES:
            raise ValueError(f"{self.exp_id}: unknown input mode {self.input_mode!r}")
        for name in ("train_domains", "select_domains"):
            domains = getattr(self, name)
            if not domains or not set(domains) <= set(DOMAINS):
                raise ValueError(f"{self.exp_id}: {name} must be a non-empty subset of {DOMAINS}, got {domains!r}")
        if set(self.select_domains) != set(self.train_domains):
            raise ValueError(
                f"{self.exp_id}: select_domains must equal train_domains (checkpoints are chosen on the domains "
                f"the model was trained on; anything else leaks information), got {self.select_domains} vs {self.train_domains}"
            )
        if not set(self.test_subsets) <= set(SUBSETS):
            raise ValueError(f"{self.exp_id}: test_subsets must be a subset of {SUBSETS}")

    def is_resolved(self) -> bool:
        return self.backbone is not None and self.input_mode is not None and self.lr is not None